In [ ]:
R.Version()

In [ ]:
library(tidyverse)

In [ ]:
library(janitor)
library(dplyr)
library(ggplot2)
library(tidyr)
library(readr)
library(tibble)
library(purrr)
library(stringr)
library(forcats)

In [ ]:
# read saved metadata
meta_final <- read.csv("251206_MetaFinal_Complete.csv")

In [ ]:
glimpse(meta_final)

In [ ]:
# basic checks
nrow(meta_final)                        # expect 616
length(unique(meta_final$sample))       # expect 616
head(meta_final)

In [ ]:
# ---- EXPRESSION MATRIX ----
expr_raw <- read.csv("20250710_RNana_Seqtk_Filtered_vst_limma_w.DesignMat.csv")

In [ ]:
dim(expr_raw)
head(expr_raw)

In [ ]:
library(readr)
library(dplyr)
library(stringr)

# 1. Load
expr_raw <- read_csv(
  "20250710_RNana_Seqtk_Filtered_vst_limma_w.DesignMat.csv",
  show_col_types = FALSE
)

# 2. Clean column names
clean_names <- function(x) {
  x %>%
    str_replace("_[Rr]edo.*$", "") %>%
    str_replace("_[Vv][0-9]+$", "") %>%
    str_replace("_[Rr][0-9]+$", "") %>%
    str_replace("_[0-9]+$", "")
}

expr_clean <- expr_raw
colnames(expr_clean) <- colnames(expr_clean) %>% clean_names()

# 3. Convert first column = gene IDs → rownames
expr_clean <- as.data.frame(expr_clean)
rownames(expr_clean) <- expr_clean[[1]]
expr_clean[[1]] <- NULL

# 4. Convert all columns to numeric → this is expr_num
expr_num <- expr_clean %>%
  mutate(across(everything(), as.numeric)) %>%
  as.matrix()

dim(expr_num)
head(expr_num[, 1:5])


In [ ]:
identical(colnames(expr_num), meta_final$sample_clean)

In [ ]:
dim(expr_num)

In [ ]:
#transpose for WGCNA

datExpr_all <- t(expr_num)   # samples × genes
dim(datExpr_all)             # 616 × 18452

## WGCNA

In [ ]:
# make sure BiocManager exists
#if (!requireNamespace("BiocManager", quietly = TRUE))
  #install.packages("BiocManager", repos="https://cloud.r-project.org")

#install the Bioconductor deps WGCNA needs
#BiocManager::install(c("impute", "preprocessCore"), update = FALSE, ask = FALSE)

In [ ]:
#install.packages("WGCNA", dependencies = TRUE)

In [ ]:
library(WGCNA)
allowWGCNAThreads()

In [ ]:
dim(datExpr_all)

In [ ]:
gsg_all <- goodSamplesGenes(datExpr_all, verbose = 3)
if (!gsg_all$allOK) {
  datExpr_all <- datExpr_all[, gsg_all$goodGenes]
  message("Removed bad genes: ", sum(!gsg_all$goodGenes))
}


In [ ]:
par(mfrow = c(1,2))
plot(hclust(dist(datExpr_all), method = "average"),
     main = "Sample clustering", xlab = "", sub = "", cex = 0.6)

In [ ]:
powers <- c(1:12, seq(14, 30, 2))

pick_soft_power <- function(datExpr, label = "dataset") {
  sft <- pickSoftThreshold(
    datExpr,
    powerVector = powers,
    corFnc      = "bicor",
    corOptions  = list(maxPOutliers = 0.1),
    verbose     = 5
  )
  
  par(mfrow = c(1,2))
  cex1 <- 0.8
  
  plot(sft$fitIndices[,1],
       -sign(sft$fitIndices[,3]) * sft$fitIndices[,2],
       xlab = "Soft Threshold (power)",
       ylab = "Scale Free Topology Model Fit, signed R^2",
       type = "n",
       main = paste("Scale independence –", label))
  text(sft$fitIndices[,1],
       -sign(sft$fitIndices[,3]) * sft$fitIndices[,2],
       labels = powers,
       cex = cex1, col = "red")
  abline(h = 0.80, col = "red", lty = 2)
  
  plot(sft$fitIndices[,1],
       sft$fitIndices[,5],
       xlab = "Soft Threshold (power)",
       ylab = "Mean connectivity",
       type = "n",
       main = paste("Mean connectivity –", label))
  text(sft$fitIndices[,1],
       sft$fitIndices[,5],
       labels = powers,
       cex = cex1, col = "red")
  
  sft
}

sft_all <- pick_soft_power(datExpr_all, label = "All samples")


In [ ]:
pick_power_from_sft <- function(sft, r2_target = 0.8) {
  fit <- sft$fitIndices
  r2  <- -sign(fit[,3]) * fit[,2]
  cand <- fit[r2 > r2_target, 1]
  if (length(cand) == 0) {
    warning("No power exceeds R² target; returning NA")
    return(NA_integer_)
  }
  min(cand)
}

softPower_all <- pick_power_from_sft(sft_all, r2_target = 0.8)
softPower_all

In [ ]:
k <- softConnectivity(datExpr_all, power = 6)

In [ ]:
par(mfrow = c(2,2))
hist(k, main = paste("connectivity (power =", softPower_all, ")"))
scaleFreePlot(k, main = "scale-free check")

In [ ]:
net_all <- blockwiseModules(
  datExpr_all,
  power = 6,
  networkType = "signed",
  corType = "bicor",
  deepSplit = 4,
  pamRespectsDendro = FALSE,
  minModuleSize = 30,        # allow more modules
  mergeCutHeight = 0.15,     # less aggressive merging
  maxBlockSize = 10000,
  saveTOMs = TRUE,
  saveTOMFileBase = "TOM_All",
  TOMType = "signed",
  numericLabels = FALSE,
  verbose = 3
)

# make the tree/dendrogram


In [ ]:
#module eigengenes

MEs_all <- orderMEs(net_all$MEs)
head(MEs_all)

In [ ]:
write.csv(MEs_all, "260211_WGCNA_MEs_2blocks.csv", row.names = TRUE)

In [ ]:
library(tibble)
library(dplyr)
library(readr)

## Check that dimensions match
length(net_all$colors)
ncol(datExpr_all)

# If datExpr_autumn is samples x genes (WGCNA default):
modules_all <- tibble(
  gene   = colnames(datExpr_all),
  module = as.character(net_all$colors)
)

# sanity checks
head(modules_all)
table(modules_all$module)


In [ ]:
genes_by_module <- split(modules_all$gene, modules_all$module)

# Example: genes in Autumn "blue" module
genes_by_module[["turquoise"]][1:10]

In [ ]:
write_csv(modules_all, "260131_WGCNA_All_gene_modules_2blocks_.csv")

In [ ]:
#module eigengene - Proportion variance modules global

In [ ]:
# ── 1. Build gene-to-module mapping ──────────────────────────
modules_all <- tibble(
  gene   = colnames(datExpr_all),
  module = as.character(net_all$colors)
)


In [ ]:
# Drop the "grey" (unassigned) module — it has no biological meaning
modules_all <- modules_all %>% filter(module != "grey")

In [ ]:
# ── 2. PCA per module: extract variance explained for PCs 1-10
n_pcs <- 10

pca_var <- modules_all %>%
  distinct(module) %>%
  pull(module) %>%
  map_dfr(function(mod) {
    genes_in_mod <- modules_all %>% filter(module == mod) %>% pull(gene)

    # Subset expression matrix to this module's genes
    expr_sub <- datExpr_all[, genes_in_mod, drop = FALSE]

    # Need at least n_pcs genes to extract that many PCs
    max_pcs <- min(n_pcs, ncol(expr_sub), nrow(expr_sub) - 1)
    if (max_pcs < 1) return(NULL)

    pca_res   <- prcomp(expr_sub, center = TRUE, scale. = FALSE)
    var_expl  <- (pca_res$sdev^2 / sum(pca_res$sdev^2))[1:max_pcs]

    tibble(
      module = mod,
      PC     = 1:max_pcs,
      prop_var = var_expl
    )
  })

In [ ]:
# ── 3. Plot ──────────────────────────────────────────────────
# Use actual WGCNA module colours as dot fills
mod_colours <- pca_var %>%
  distinct(module) %>%
  pull(module)

# WGCNA module names ARE their colours, so use them directly
colour_map <- setNames(mod_colours, mod_colours)

p <- ggplot(pca_var, aes(x = factor(PC), y = prop_var, colour = module)) +
  geom_jitter(size = 3, alpha = 0.85, width = 0.2, height = 0) +
  scale_colour_manual(values = colour_map) +
  scale_x_discrete(breaks = 1:n_pcs) +
  labs(
    x = "Principal Component ",
    y = "Proportion of variance explained"
  ) +
  theme_bw(8) +
  theme(
    legend.position = "none",
    panel.grid.major.x = element_blank()
  )
p



In [ ]:
# ── 4. Save ──────────────────────────────────────────────────
ggsave("module_pca_variance.pdf", p, width = 184, height = 92, unit = "mm")

message("Saved: module_pca_variance.pdf / .png")

In [ ]:
# Abiotic (weather)
abiotic_vars <- c("mean_temp_4d", "mean_hum_4d",
                  "mean_fluctuation_4d", "precip_30d_sum", "hours_after_sunrise")

# Plant morphology
morpho_vars <- c("max_diam_mm", "n_leaves")

# Presence absence

p_a_vars <- c("bolting","hr")

# Composite field scores
composite_vars <- c("composite_bacterial", "composite_fungi_hpa", "herbivory")

# Microbial load (metagenomic)
microbe_vars <- c("bacterial_per_plant", "fungal_per_plant", "bact_family_entropy")

# Field symptoms
symptom_vars <- c("herbivory")

In [ ]:
names(meta_final)

In [ ]:
# all
traits_all <- meta_final[, ] %>%
  dplyr::select(all_of(c(abiotic_vars, morpho_vars, p_a_vars,
                  composite_vars, microbe_vars, symptom_vars))) %>%
  as.data.frame()

In [ ]:
names(traits_all)

In [ ]:
library(WGCNA)

# recompute eigengenes with varExpl (if not already)
MElist_all <- moduleEigengenes(datExpr_all, colors = net_all$colors)

# Autumn
me_names_all   <- names(MElist_all$varExpl)     # "MEturquoise", ...
mod_colors_all <- sub("^ME", "", me_names_all)   # "turquoise", ...

scree_all <- tibble(
  ModuleName = me_names_all,
  Color      = mod_colors_all,
  VarExpl    = as.numeric(MElist_all$varExpl) * 100,  # % variance
)

In [ ]:
p_ME_PC1_simple <- ggplot(scree_all,
                          aes(x = ModuleName, y = VarExpl, colour = ModuleName)) +
  geom_point(position = position_jitter(width = 0.15, height = 0),
             alpha = 0.9, size = 2.4) +
  labs(
    title = "Variance explained by ME1 per module",
    x     = "Module",
    y     = "Variance explained by ME1 (%)"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    axis.text.x        = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 7),
    panel.grid.major.x = element_blank(),
    legend.position    = "none"
  )

p_ME_PC1_simple


In [ ]:
#module-trait correlations

cor_all <- cor(MEs_all, traits_all, use = "pairwise.complete.obs")
p_all   <- corPvalueStudent(cor_all, nrow(datExpr_all))

In [ ]:
# traits for Autumn and Spring (if not yet):
trait_vars <- c(abiotic_vars, morpho_vars, p_a_vars,
                composite_vars, microbe_vars)

traits_all <- meta_final[, trait_vars] %>% as.data.frame()

# module eigengenes (if not yet):
MEs_all <- orderMEs(net_all$MEs)

# correlations + p-values
cor_all <- cor(MEs_all, traits_all, use = "pairwise.complete.obs")
p_all   <- corPvalueStudent(cor_all, nrow(datExpr_all))


In [ ]:
summary(as.vector(p_all))

In [ ]:

make_heat_df <- function(cor_mat, p_mat, season_label) {
  as.data.frame(cor_mat) %>%
    rownames_to_column("module") %>%              # modules = rows of cor_mat
    pivot_longer(-module, names_to = "trait", values_to = "cor") %>%
    mutate(
      p      = as.vector(p_mat),
    )
}

heat_all <- make_heat_df(cor_all, p_all, "All")


In [ ]:
heat_all <- heat_all %>%
mutate(p_adj = p.adjust(p, method= "BH"))

In [ ]:
sig_fdr <- heat_all %>%
filter(p_adj > 0.05)

In [ ]:
sig_fdr <- heat_all #%>%
#filter(p<0.05, abs(cor) >= 0.2) %>%
#arrange(p)

In [ ]:
sig_fdr

In [ ]:
write.csv(sig_fdr, "WGCNA_Mod_Trait_pvalues.csv")

In [ ]:
label_thresh <- 0.2

make_module_trait_heatmap_clean <- function(df_heat, module_order = NULL) {
  if (is.null(module_order)) {
    module_order <- unique(df_heat$module)
  }
  
  ggplot(df_heat, aes(x = module, y = trait, fill = cor)) +
    geom_tile(color = "white") +
    geom_text(
      data = subset(df_heat, abs(cor) >= label_thresh),
      aes(label = sprintf("%.2f", cor)),
      size = 1.5,
      colour = "black"
    ) +
    scale_fill_gradient2(
      low    = "#2166ac",
      mid    = "white",
      high   = "#b2182b",
      limits = c(-1, 1),
      name   = "Correlation"
    ) +
    scale_x_discrete(limits = module_order) +
    theme_bw(base_size = 9) +
    theme(
      axis.text.x  = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 7),
      axis.text.y  = element_text(size = 7),
      axis.title   = element_blank(),
      panel.grid   = element_blank(),
      strip.text   = element_text(size = 10, face = "bold"),
      legend.position = "right",
      legend.title = element_text(size = 9),
      legend.text  = element_text(size = 8)
    )
}


In [ ]:
mod_order_all <- colnames(MEs_all)

p_mt_all <- make_module_trait_heatmap_clean(
  heat_all,
  module_order = mod_order_all
) + ggtitle("All")

p_mt_all

In [ ]:
## Save as PNG (for drafts / slides)
ggsave(
  filename = "251218_Fig3D_Modules_TraitRelationships_All_2_blocks.png",
  plot     = p_mt_all,
  width    = 5,
  height   = 5,
  dpi      = 300
)


In [ ]:
# MEs_all = module eigengenes matrix (samples x modules)

ME_cor <- cor(MEs_all, use = "pairwise.complete.obs")
ME_diss <- as.dist(1 - ME_cor)
ME_tree <- hclust(ME_diss, method = "average")

module_order <- ME_tree$labels[ME_tree$order]


In [ ]:
module_colors <- data.frame(
  module = module_order,
  color  = module_order
)


In [ ]:
#install.packages("patchwork")

In [ ]:
library(ggplot2)
library(ggnewscale)
library(patchwork)

In [ ]:
make_module_trait_heatmap_with_strip <- function(df_heat,
                                                 module_order = NULL,
                                                 label_thresh = 0.2) {

  # clean module names if they are like "MEblack"
  df_heat$module <- gsub("^ME", "", df_heat$module)

  if (is.null(module_order)) {
    module_order <- unique(df_heat$module)
  } else {
    module_order <- gsub("^ME", "", module_order)
  }

  df_heat$module <- factor(df_heat$module, levels = module_order)

  # --- strip data ---
  strip_df <- data.frame(
    module = factor(module_order, levels = module_order),
    y = 1
  )

  # Top strip (module colors)
  p_strip <- ggplot(strip_df, aes(x = module, y = y, fill = as.character(module))) +
    geom_tile() +
    scale_fill_identity() +  # because module names are actual color names
    scale_x_discrete(drop = FALSE) +
    theme_void(base_size = 9) +
    theme(
      plot.margin = margin(0, 4, 0, 4),
      axis.text.x = element_blank()
    )


  # Main heatmap (correlations)
  p_heat <- ggplot(df_heat, aes(x = module, y = trait, fill = cor)) +
    geom_tile(color = "white") +
    geom_text(
      data = subset(df_heat, abs(cor) >= label_thresh),
      aes(label = sprintf("%.2f", cor)),
      size = 1.5,
      colour = "black"
    ) +
    scale_fill_gradient2(
      low    = "#2166ac",
      mid    = "white",
      high   = "#b2182b",
      limits = c(-1, 1),
      name   = "Correlation"
    ) +
    scale_x_discrete(drop = FALSE, labels = NULL) +
    theme_bw(base_size = 9) +
    theme(
      axis.text.y  = element_text(size = 7),
      axis.title   = element_blank(),
      panel.grid   = element_blank(),
      plot.margin  = margin(0, 4, 4, 4),
      legend.position = "right"
    )

  # Stack strip + heatmap (strip is short)
p_heat / p_strip + plot_layout(heights = c(1, 0.06))
}


In [ ]:
# Example: module_order from eigengene clustering (recommended)
ME_cor <- cor(MEs_all, use = "pairwise.complete.obs")
ME_tree <- hclust(as.dist(1 - ME_cor), method = "average")
module_order <- gsub("^ME", "", ME_tree$labels[ME_tree$order])

p_mt_all <- make_module_trait_heatmap_with_strip(
  heat_all,
  module_order = module_order,
  label_thresh = 0.2
)

p_mt_all


In [ ]:
# --- module order from eigengene clustering ---
ME_cor <- cor(MEs_all, use = "pairwise.complete.obs")
ME_tree <- hclust(as.dist(1 - ME_cor), method = "average")
module_order <- gsub("^ME", "", ME_tree$labels[ME_tree$order])

# --- trait order from correlation profile clustering ---
df_heat2 <- heat_all
df_heat2$module <- gsub("^ME", "", df_heat2$module)

# wide matrix trait x module of correlations
mat_tm <- xtabs(cor ~ trait + module, data = df_heat2)

trait_tree  <- hclust(dist(mat_tm), method = "average")
trait_order <- rownames(mat_tm)[trait_tree$order]


In [ ]:
plot_module_trait_heatmap_B <- function(df_heat,
                                        module_order,
                                        trait_order,
                                        label_thresh = 0.2) {

  df_heat$module <- gsub("^ME", "", df_heat$module)

  df_heat$module <- factor(df_heat$module, levels = module_order)
  df_heat$trait  <- factor(df_heat$trait,  levels = trait_order)

  # Bottom strip = one tile per module, filled with the module color name
  strip_df <- data.frame(
    module = factor(module_order, levels = module_order),
    y = 1
  )

  p_strip <- ggplot(strip_df, aes(x = module, y = y, fill = as.character(module))) +
    geom_tile() +
    scale_fill_identity() +
    scale_x_discrete(drop = FALSE) +
    theme_void(base_size = 8) +
    theme(
      plot.margin = margin(0, 4, 0, 4)
    )

  p_heat <- ggplot(df_heat, aes(x = module, y = trait, fill = cor)) +
    geom_tile(color = "white") +
    geom_text(
      data = subset(df_heat, abs(cor) >= label_thresh),
      aes(label = sprintf("%.2f", cor)),
      size = 1.5,
      colour = "black"
    ) +
    scale_fill_gradient2(
      low    = "#2166ac",
      mid    = "white",
      high   = "#b2182b",
      limits = c(-1, 1),
      name   = "Correlation"
    ) +
    scale_x_discrete(drop = FALSE, labels = NULL) +
    theme_bw(base_size = 8) +
    theme(
      panel.grid   = element_blank(),
      axis.title   = element_blank(),
      axis.text.y  = element_text(size = 6),
      axis.text.x  = element_blank(),
      plot.margin  = margin(4, 4, 0, 4),
      legend.position = "right",
      legend.title = element_text(size = 8),
      legend.text  = element_text(size = 7)
    )

  # Heatmap on top, strip at bottom
  p_heat / p_strip + plot_layout(heights = c(1, 0.06))
}


In [ ]:
p_mt_all_B <- plot_module_trait_heatmap_B(
  df_heat      = heat_all,
  module_order = module_order,
  trait_order  = trait_order,
  label_thresh = 0.2
)

p_mt_all_B


In [ ]:
ggsave("Fig2E_module_trait_heatmap_clustered_2block.pdf",
       plot   = p_mt_all_B,
       width  = 184,
       height = 184,     
       units  = "mm",
       device = cairo_pdf)


In [ ]:
library(dplyr)
library(ggplot2)
library(forcats)

# Gene → module mapping per season
module_all <- data.frame(
  gene   = names(net_all$colors),
  module = net_all$colors,
  row.names = NULL
)


# Module sizes
modsizes_all <- module_all %>%
  count(module, name = "n_genes") %>%
  mutate(season = "All")

In [ ]:
modsizes_all <- modsizes_all %>%
  mutate(module = fct_reorder(module, n_genes, .desc = TRUE)) %>%
  ungroup()

p_modsizes <- ggplot(modsizes_all,
                     aes(x = module, y = n_genes, fill = module)) +
  geom_col(colour = "black", linewidth = 0.2) +
  facet_wrap(~ season, scales = "free_x") +
  scale_fill_manual(values = levels(as.factor(modsizes_all$module)),
                    guide = "none") +  # WGCNA colors already meaningful
  theme_bw(base_size = 10) +
  theme(
    axis.text.x  = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 7),
    panel.grid   = element_blank(),
    strip.text   = element_text(size = 11, face = "bold")
  ) +
  labs(
    title = "",
    x     = "Module (color)",
    y     = "Number of genes"
  )

p_modsizes


In [ ]:
library(dplyr)
library(tidyr)

In [ ]:
cor_all

In [ ]:
## ─────────────────────────────────────────────────────────────
## Module eigengene correlation network — pure base R
## No igraph, ggraph, tidyverse, or clusterProfiler required
## Requires: MEs_all, net_all, datExpr_all (from your WGCNA run)
## ─────────────────────────────────────────────────────────────

# ── 0. Parameters ────────────────────────────────────────────
thresh       <- 0.3          # |r| threshold to draw an edge
node_scale   <- 0.8          # tune overall node size
edge_lwd_max <- 4            # max edge line width
layout_type  <- "circle"     # "circle" or "fr" (Fruchterman-Reingold)

# ── 1. Correlation matrix of module eigengenes ───────────────
ME_cor  <- cor(MEs_all, use = "pairwise.complete.obs")
mod_names <- colnames(MEs_all)                     # e.g. "MEblue", "MEbrown"
mod_colors <- sub("^ME", "", mod_names)            # strip "ME" prefix → colour name
n_mods <- length(mod_names)

In [ ]:
# ── 2. Edge list ─────────────────────────────────────────────
edges <- data.frame(from = integer(), to = integer(),
                    cor = numeric(), stringsAsFactors = FALSE)
for (i in 1:(n_mods - 1)) {
  for (j in (i + 1):n_mods) {
    r <- ME_cor[i, j]
    if (abs(r) >= thresh) {
      edges <- rbind(edges, data.frame(from = i, to = j, cor = r))
    }
  }
}

In [ ]:
# ── 3. Node sizes — proportional to number of genes ─────────
gene_counts <- table(as.character(net_all$colors))
# match order to MEs
node_sizes <- as.numeric(gene_counts[mod_colors])
node_sizes[is.na(node_sizes)] <- 1
# scale to plottable range
node_sizes_scaled <- sqrt(node_sizes) / max(sqrt(node_sizes)) * 6 * node_scale

In [ ]:
# ── 4. Layout ────────────────────────────────────────────────

circle_layout <- function(n) {
  angles <- seq(0, 2 * pi, length.out = n + 1)[1:n]
  cbind(x = cos(angles), y = sin(angles))
}

# Simple Fruchterman–Reingold in base R (no dependencies)
fr_layout <- function(adj, n_iter = 300) {
  n <- nrow(adj)
  pos <- matrix(runif(n * 2, -1, 1), ncol = 2)
  area <- 4; k <- sqrt(area / n)
  temp <- 1.0
  for (iter in seq_len(n_iter)) {
    disp <- matrix(0, n, 2)
    # repulsion
    for (i in 1:n) {
      for (j in 1:n) {
        if (i == j) next
        delta <- pos[i, ] - pos[j, ]
        d <- max(sqrt(sum(delta^2)), 0.01)
        disp[i, ] <- disp[i, ] + (delta / d) * (k^2 / d)
      }
    }
    # attraction (edges only)
    for (i in 1:n) {
      for (j in 1:n) {
        if (i >= j || adj[i, j] == 0) next
        delta <- pos[i, ] - pos[j, ]
        d <- max(sqrt(sum(delta^2)), 0.01)
        force <- (d^2 / k) * delta / d
        disp[i, ] <- disp[i, ] - force
        disp[j, ] <- disp[j, ] + force
      }
    }
    # limit displacement by temperature
    for (i in 1:n) {
      d <- max(sqrt(sum(disp[i, ]^2)), 0.01)
      pos[i, ] <- pos[i, ] + (disp[i, ] / d) * min(d, temp)
    }
    temp <- temp * (1 - iter / n_iter)
  }
  # normalise to [-1, 1]
  pos[,1] <- (pos[,1] - min(pos[,1])) / max(diff(range(pos[,1])), 0.01) * 2 - 1
  pos[,2] <- (pos[,2] - min(pos[,2])) / max(diff(range(pos[,2])), 0.01) * 2 - 1
  pos
}

if (layout_type == "fr") {
  adj_mat <- (abs(ME_cor) >= thresh) * 1; diag(adj_mat) <- 0
  lay <- fr_layout(adj_mat)
} else {
  lay <- circle_layout(n_mods)
}

In [ ]:
# ── 5. Plotting function ─────────────────────────────────────
# Wrapped so all draw calls hit the same device (avoids
# "plot.new has not been called yet" in notebooks / RStudio)

draw_network <- function() {
  par(mar = c(2, 1, 2, 1), xpd = TRUE)
  plot(lay, type = "n", asp = 1,
       xlim = c(-1.4, 1.4), ylim = c(-1.4, 1.4),
       axes = FALSE, xlab = "", ylab = "")

  # Draw edges
  if (nrow(edges) > 0) {
    for (e in seq_len(nrow(edges))) {
      i   <- edges$from[e]
      j   <- edges$to[e]
      r   <- edges$cor[e]
      lwd <- (abs(r) - thresh) / (1 - thresh) * edge_lwd_max + 0.3
      lty <- if (r > 0) 1 else 2            # solid = positive, dashed = negative
      segments(lay[i, 1], lay[i, 2],
               lay[j, 1], lay[j, 2],
               col = "grey40", lwd = lwd, lty = lty)
    }
  }

  # Draw nodes
  points(lay, pch = 21,
         bg  = mod_colors,
         col = "black",
         cex = node_sizes_scaled,
         lwd = 0.8)

  # Labels
  text(lay[, 1], lay[, 2] - node_sizes_scaled * 0.04 - 0.10,
       labels = mod_colors,
       cex = 0.55, font = 2)

  # Legend
  legend("bottomright",
         legend = c("Positive edge", "Negative edge"),
         lty = c(1, 2), lwd = c(2, 2), col = "grey40",
         bty = "n", cex = 0.6)

  # Annotation
  mtext(paste0("V=", n_mods, "  E=", nrow(edges),
               "  |r| threshold=", thresh),
        side = 1, line = 0.5, cex = 0.55)
  title("Module Eigengene Network", cex.main = 0.8)
}


In [ ]:
# ── 6. Save to PDF ───────────────────────────────────────────
pdf("260211_ME_network_all.pdf", width = 184 / 25.4, height = 184 / 25.4)
draw_network()
dev.off()

# ── 7. Also draw to screen (notebook / RStudio) ─────────────
draw_network()

message("Saved: ME_network_all.pdf")

## hub genes

In [ ]:
library(WGCNA)
library(dplyr)
library(tibble)

# Just to be safe
allowWGCNAThreads()

make_hub_table <- function(datExpr, net, MEs, softPower) {
  # datExpr: samples x genes
  # net: output of blockwiseModules
  # MEs: module eigengenes (orderMEs)
  # softPower: chosen soft-thresholding power
  
  # 1) module colors for each gene
  module_colors <- as.character(net$colors)
  genes         <- colnames(datExpr)
  
  stopifnot(length(genes) == length(module_colors))
  
  # 2) intramodular connectivity (kWithin) using expression directly
  kWithin_df <- intramodularConnectivity.fromExpr(
    datExpr,
    colors = module_colors,
    power  = softPower,
    networkType = "signed"  # same as your network
  )
  # kWithin is in column "kWithin"
  kWithin <- kWithin_df$kWithin
  
  # 3) module membership (kME) = cor(gene, module eigengenes)
  kME_all <- signedKME(
    datExpr,
    MEs,
    outputColumnName = "ME"  # columns like "MEblue", "MEbrown", ...
  )
  
  # For each gene, grab its kME to *its own* module
  # Example: gene in "blue" → use column "MEblue"
  kME_in_module <- sapply(seq_along(genes), function(i) {
    this_mod <- module_colors[i]
    colname  <- paste0("ME", this_mod)
    if (colname %in% colnames(kME_all)) {
      kME_all[i, colname]
    } else {
      NA_real_
    }
  })
  
  # 4) Build tidy hub table
  hub_tbl <- tibble(
    gene           = genes,
    module         = module_colors,
    kWithin        = kWithin,
    kME            = kME_in_module,
  ) %>%
    group_by(module) %>%
    arrange(desc(kWithin), desc(kME), .by_group = TRUE) %>%
    mutate(rank_in_module = row_number()) %>%
    ungroup()
  
  hub_tbl
}


In [ ]:
hubs_all <- make_hub_table(
  datExpr    = datExpr_all,
  net        = net_all,
  MEs        = MEs_all,
  softPower  = softPower_all
)

# Quick sanity check
hubs_all %>% count(module)

In [ ]:
top10_all <- hubs_all %>%
  group_by(module) %>%
  slice_head(n = 10) %>%
  ungroup()

head(top10_all)

In [ ]:
#  compact table with top 5 hub genes per module
hub_summary_all <- hubs_all %>%
  group_by(module) %>%
  slice_head(n = 5) %>%
  summarise(
    top_hubs = paste(gene, collapse = ", "),
    .groups  = "drop"
  )

hub_summary_all

In [ ]:
library(dplyr)
library(tidyr)

## 1) Take the top N hubs per module
topN <- 10

top_hubs_all <- hubs_all %>%
  group_by(module) %>%
  slice_head(n = topN) %>%
  ungroup() 

head(top_hubs_all)

# GO terms

In [ ]:
library(tibble)
library(dplyr)

## Check what your expression matrices look like
dim(datExpr_all)
head(colnames(datExpr_all))   # should be gene IDs (e.g. AT1G01010)
head(rownames(datExpr_all))   # should be samples

## IMPORTANT: colnames(datExpr_*) must be your gene IDs.
## If they are NULL or just "V1", "V2", we need to fix that from the
## original expression matrix before WGCNA / GO.


In [ ]:
library(dplyr)
library(clusterProfiler)
library(org.At.tair.db)   # Arabidopsis
library(ggplot2)
library(ggrepel)
library(enrichplot)

In [ ]:
library(tibble)
library(dplyr)

In [ ]:
# --- All ---
modules_all<- tibble(
  gene   = colnames(datExpr_all),      # gene IDs (should be ATxGxxxxx)
  module = as.character(net_all$colors)
)
head(modules_all)
table(modules_all$module)


In [ ]:
table(modules_all$module)

In [ ]:
names(modules_all)

In [ ]:
library(dplyr)
library(stringr)
library(clusterProfiler)
library(org.At.tair.db)
library(ggplot2)
library(ggrepel)
library(forcats)
library(purrr)
library(tidyr)

# Clean gene IDs (remove things like ".1" if present)
modules_all <- modules_all %>%
  mutate(
    gene_tair = str_replace(gene, "\\.\\d+$", "")
  )

# Optional: drop "grey" (unassigned) modules
#modules_autumn <- modules_autumn %>% filter(module != "grey")
#modules_spring <- modules_spring %>% filter(module != "grey")


In [ ]:
run_go_for_modules <- function(mod_tab,
                               min_genes = 10,
                               p_cutoff = 0.05) {
  mod_list <- split(mod_tab$gene_tair, mod_tab$module)

  go_list <- imap(mod_list, function(genes, mod_name) {
    genes <- unique(genes)
    if (length(genes) < min_genes) {
      message("Skipping module ", mod_name, " (", length(genes), " genes)")
      return(NULL)
    }

    message("Running GO BP for module ", mod_name,
            " (", length(genes), " genes)")

    enrichGO(
      gene          = genes,
      OrgDb         = org.At.tair.db,
      keyType       = "TAIR",
      ont           = "BP",
      pAdjustMethod = "BH",
      pvalueCutoff  = p_cutoff,
      readable      = TRUE
    )
  })

  # drop modules that had NULL (too few genes or no enrichment)
  go_list[!vapply(go_list, is.null, logical(1))]
}

go_all<- run_go_for_modules(modules_all)

In [ ]:
names(go_all)
go_all[["turquoise"]]   # example module

In [ ]:
tidy_go_results <- function(go_list, top_n = 20) {
  imap_dfr(go_list, function(res, mod_name) {
    if (is.null(res) || nrow(as.data.frame(res)) == 0) return(NULL)

    as_tibble(res@result) %>%
      arrange(p.adjust) %>%
      slice_head(n = top_n) %>%
      mutate(module = mod_name)
  })
}

go_all_tbl <- tidy_go_results(go_all, top_n = 20)

# quick peek
go_all_tbl %>% count(module)

In [ ]:
plot_module_go <- function(go_res, module_name, top_n = 15) {
  if (is.null(go_res) || nrow(as.data.frame(go_res)) == 0) {
    stop("No enrichment results for module ", module_name)
  }

  df <- as_tibble(go_res@result) %>%
    arrange(p.adjust) %>%
    slice_head(n = top_n) %>%
    # turn GeneRatio "12/200" into numeric
    separate(GeneRatio, into = c("k","n"), sep = "/", convert = TRUE) %>%
    mutate(GeneRatio_num = k / n,
           Description = fct_reorder(Description, GeneRatio_num))

  ggplot(df,
         aes(x = GeneRatio_num,
             y = Description,
             size = Count,
             colour = p.adjust)) +
    geom_point() +
    scale_colour_viridis_c(direction = -1, option = "magma") +
    labs(
      title = paste0("GO BP enrichment – module ", module_name),
      x     = "Gene ratio",
      y     = NULL,
      colour = "adj. p-value",
      size   = "Gene count"
    ) +
    theme_minimal(base_size = 11) +
    theme(
      plot.title = element_text(size = 13, face = "bold"),
      axis.text.y = element_text(size = 8),
      legend.position = "right"
    )
}


In [ ]:
#
p_blue_all <- plot_module_go(go_all[["black"]], "black", top_n = 15)
p_blue_all

In [ ]:
write.csv(go_all_tbl,
         file = "251210_GO_enrichment_All_modules.csv",
         row.names = FALSE)

In [ ]:
library(dplyr)
library(tidyr)

# make sure these exist:
# go_autumn_tbl, go_spring_tbl

go_all_ranked <- go_all_tbl %>%
  group_by(module) %>%
  arrange(p.adjust, .by_group = TRUE) %>%
  mutate(rank = row_number()) %>%
  ungroup()


In [ ]:
# helper: collapse top N descriptions into one string
summarise_top_terms <- function(df, top_n = 3) {
  df %>%
    group_by(module) %>%
    arrange(p.adjust, .by_group = TRUE) %>%
    summarise(
      n_terms = n(),
      top_terms = paste(head(Description, top_n), collapse = "; "),
      .groups = "drop"
    )
}

all_summary <- summarise_top_terms(go_all_tbl, top_n = 3) %>%
  dplyr::select(module, n_terms, top_terms)

all_summary

In [ ]:
make_wide_top_terms <- function(df, top_n = 3) {
  df %>%
    group_by(module) %>%
    arrange(p.adjust, .by_group = TRUE) %>%
    slice_head(n = top_n) %>%
    mutate(term_rank = paste0("TopTerm_", row_number())) %>%
    dplyr::select(module, term_rank, Description) %>%
    pivot_wider(
      names_from  = term_rank,
      values_from = Description
    )
}

all_top_wide <- make_wide_top_terms(go_all_tbl, top_n = 3)

top_terms_wide_all <- all_top_wide

top_terms_wide_all


In [ ]:
write.csv(top_terms_wide_all,
          "251210_GO_BP_module_summary_compact.csv",
          row.names = FALSE)

In [ ]:
#done